# ODI to Databricks Migration: TRG_LOC Full Load

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook performs a full load insert into the `workspace.hr.trg_loc` table from `workspace.hr.locations`.
The original ODI SQL represented a simple data transfer without complex transformations or incremental logic. Oracle `/*+ APPEND PARALLEL */` hints have been removed as they are not applicable to Databricks Delta Lake.

**Source ODI `SCEN_TASK_NO` References:**
- `{10}`: Start of load process
- `{20}`: Step marker
- `{30}`: Step marker, followed by INSERT statement

In [ ]:
import datetime

# Create boilerplate widgets as per standard ETL framework
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., 'F' for Full, 'I' for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "2. Data Source Numeric ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "3. ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "4. ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", datetime.datetime(1900, 1, 1).strftime("%Y-%m-%d %H:%M:%S"), "5. Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "6. Current Extract Time (YYYY-MM-DD HH:MM:SS)")

# ETL Parameters

The following parameters are typically passed to the ETL process. While this specific SQL does not use them directly, they are included for completeness of the ETL framework.

In [ ]:
display(spark.sql(f"""
  SELECT
    '${ETL_JOB_TYPE}' AS etl_job_type,
    ${DATASOURCE_NUM_ID} AS datasource_num_id,
    ${ETL_PROC_WID} AS etl_proc_wid,
    '${ODI_SESS_NO}' AS odi_sess_no,
    to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
    to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time
"""))

# Target Load: `TRG_LOC` (SCEN_TASK_NO {10} - {30})

This section performs a full data load from the source `LOCATIONS` table into the `TRG_LOC` target table. 

In [ ]:
%sql
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID,
  STREET_ADDRESS,
  POSTAL_CODE,
  CITY,
  STATE_PROVINCE,
  COUNTRY_ID
)
SELECT
  locations.LOCATION_ID,
  locations.STREET_ADDRESS,
  locations.POSTAL_CODE,
  locations.CITY,
  locations.STATE_PROVINCE,
  locations.COUNTRY_ID
FROM
  workspace.hr.locations AS locations;

# Conversion Notes and Manual Actions Required

1.  **Oracle Hints Removed**: The Oracle hints `/*+ APPEND PARALLEL */` were removed as they are specific to Oracle's SQL engine and not applicable in Databricks Spark SQL / Delta Lake.
2.  **Schema and Table Naming**: Oracle schema `HR` has been converted to `workspace.hr` and table names `TRG_LOC`, `LOCATIONS` to lowercase `trg_loc`, `locations` following Databricks naming conventions.
3.  **Data Types**: Since the original DDL for `TRG_LOC` and `LOCATIONS` was not provided, the column types in the `SELECT` statement are assumed to map directly based on typical usage (e.g., `LOCATION_ID` as `BIGINT`, address fields as `STRING`, etc.). Ensure the actual target table `workspace.hr.trg_loc` has matching or compatible data types. For instance, `LOCATION_ID` and `COUNTRY_ID` if numeric, should typically be `BIGINT` in Spark SQL for Oracle `NUMBER` types with precision 0.
4.  **No Complex Logic**: This particular ODI task involved a straightforward `INSERT SELECT`. No complex MERGE, UPDATE, DELETE, or staging table logic was present in the provided snippet.